In [1]:
import os
import pandas as pd
import torch
import torch.nn as nn
import timm
import numpy as np
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

/opt/conda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0))

torch.backends.cudnn.benchmark = True

Device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


In [3]:
class AIGenDetDataset(Dataset):

    def __init__(self, shard_path, transform=None):

        self.shard_path = shard_path
        self.transform = transform

        self.labels = pd.read_csv(
            os.path.join(shard_path, "labels.csv")
        )

        print(f"Loaded {len(self.labels)} images")


    def __len__(self):
        return len(self.labels)


    def __getitem__(self, idx):

        row = self.labels.iloc[idx]

        img_path = os.path.join(
            self.shard_path,
            "images",
            row["image_name"]
        )

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = torch.tensor(row["label"]).long()

        return image, label

In [4]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(380),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(5),
    transforms.ColorJitter(0.1,0.1,0.1,0.05),
    transforms.ToTensor(),
])

val_transform = transforms.Compose([
    transforms.Resize(400),
    transforms.CenterCrop(380),
    transforms.ToTensor(),
])

In [5]:
import os

print(os.path.exists("/home/daniel"))
print(os.path.exists("/home/daniel/datasets"))
print(os.path.exists("/home/daniel/datasets/ai-gen"))

False
False
False


In [5]:
# DATASET_PATH = "/mnt/c/Development/ai-gen-detection/shard_0/shard_0"
DATASET_PATH = "/workspace/datasets/ai-gen/mnt/c/Development/ai-gen-detection/shard_0/shard_0"

dataset = AIGenDetDataset(
    DATASET_PATH
)

Loaded 50000 images


In [7]:
img, label = dataset[0]

print("Image shape:", img.size)
print("Label:", label)

Image shape: (3456, 2736)
Label: tensor(0)


In [8]:
# import time

# start = time.time()

# for i in range(2000):
#     _ = dataset[i]

# print("Time:", time.time() - start)

In [6]:
# Create custom dataset classes for train and val with proper transforms
class TransformedDataset(torch.utils.data.Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform
    
    def __getitem__(self, idx):
        image, label = self.subset[idx]
        if self.transform:
            image = self.transform(image)
        return image, label
    
    def __len__(self):
        return len(self.subset)

indices = list(range(len(dataset)))

train_idx, val_idx = train_test_split(
    indices,
    test_size=0.1,
    random_state=42,
    stratify=dataset.labels["label"]
)

# Create subsets and wrap with proper transforms
train_subset = torch.utils.data.Subset(dataset, train_idx)
val_subset = torch.utils.data.Subset(dataset, val_idx)

train_dataset = TransformedDataset(train_subset, train_transform)
val_dataset = TransformedDataset(val_subset, val_transform)

In [18]:
BATCH_SIZE = 16  # Optimized for RTX 4060 laptop (8GB VRAM)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,      # Multi-worker data loading for better performance
    pin_memory=True,
    prefetch_factor=2   # Prefetch 2 batches per worker
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE * 2,  # Can use larger batch for validation (no gradients)
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    prefetch_factor=2
)

In [11]:
# %pip install --upgrade filelock

In [8]:
model = timm.create_model(
    "efficientnet_b4",
    pretrained=True,
    num_classes=2
)

model = model.to(device)

# model = torch.compile(model)

In [9]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)

# Add learning rate scheduler for better convergence
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=5,
    eta_min=1e-6
)

scaler = torch.cuda.amp.GradScaler()

In [10]:
# Clear GPU cache and check memory before training setup
torch.cuda.empty_cache()

print("=== GPU Memory Status ===")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
print(f"Allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
print(f"Reserved: {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")
print(f"Free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_reserved(0)) / 1024**3:.2f} GB")

=== GPU Memory Status ===
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
Total VRAM: 8.00 GB
Allocated: 0.07 GB
Reserved: 0.08 GB
Free: 7.91 GB


In [11]:
def train_epoch(loader):

    model.train()

    running_loss = 0

    for images, labels in tqdm(loader):

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)  # More efficient than zero_grad()

        with torch.amp.autocast('cuda'):

            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()

    return running_loss / len(loader)

In [16]:
def validate(loader, return_details=False):

    model.eval()

    correct = 0
    total = 0
    
    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():

        for images, labels in tqdm(loader):

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.amp.autocast('cuda'):
                outputs = model(images)

            preds = torch.argmax(outputs, dim=1)
            probs = torch.softmax(outputs, dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)
            
            if return_details:
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
                all_probs.extend(probs.cpu().numpy())

    accuracy = correct / total
    
    if return_details:
        return accuracy, all_preds, all_labels, all_probs
    return accuracy

In [18]:
EPOCHS = 5

for epoch in range(EPOCHS):

    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    train_loss = train_epoch(train_loader)

    val_acc = validate(val_loader)

    print("Train Loss:", train_loss)
    print("Validation Accuracy:", val_acc)
    print("Learning Rate:", optimizer.param_groups[0]['lr'])
    print(f"GPU Memory: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
    
    # Step the scheduler
    scheduler.step()
    
    # Clear cache after each epoch
    torch.cuda.empty_cache()


Epoch 1/5


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 157/157 [01:17<00:00,  2.03it/s]


Train Loss: 0.27100760726754114
Validation Accuracy: 0.9548
Learning Rate: 0.0003
GPU Memory: 0.28 GB

Epoch 2/5


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 157/157 [01:13<00:00,  2.13it/s]


Train Loss: 0.15416599922160257
Validation Accuracy: 0.9582
Learning Rate: 0.0002714480406590546
GPU Memory: 0.28 GB

Epoch 3/5


...

Train Loss: 0.10642446386583157
Validation Accuracy: 0.9618
Learning Rate: 0.00019669804065905458
GPU Memory: 0.28 GB

Epoch 4/5


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 157/157 [01:12<00:00,  2.17it/s]


Train Loss: 0.07689681921751526
Validation Accuracy: 0.9604
Learning Rate: 0.00010430195934094532
GPU Memory: 0.28 GB

Epoch 5/5


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 157/157 [01:12<00:00,  2.17it/s]

Train Loss: 0.05395976254578125
Validation Accuracy: 0.9666
Learning Rate: 2.955195934094536e-05
GPU Memory: 0.28 GB


In [13]:
import os

MODEL_PATH = "/workspace/ai-gen-detection/training/baseline/efficientnet_b4_baseline.pth"

if os.path.exists(MODEL_PATH):
    print(f"Model file '{MODEL_PATH}' already exists. Loading model...")
    model.load_state_dict(torch.load(MODEL_PATH))
    model.to(device)
    model.eval()
    print("Model loaded successfully from disk.")
else:
    print(f"Model file not found. Saving trained model to '{MODEL_PATH}'...")
    torch.save(model.state_dict(), MODEL_PATH)
    print("Model saved successfully.")

Model file '/workspace/ai-gen-detection/training/baseline/efficientnet_b4_baseline.pth' already exists. Loading model...
Model loaded successfully from disk.


In [ ]:
# Comprehensive Model Evaluation with Metrics

print("=" * 60)
print("COMPREHENSIVE MODEL EVALUATION")
print("=" * 60)

# Get predictions and probabilities
print("\nComputing predictions on validation set...")
val_acc, preds, labels, probs = validate(val_loader, return_details=True)

# Convert to numpy arrays
preds = np.array(preds)
labels = np.array(labels)
probs = np.array(probs)

# Extract probabilities for the positive class (class 1)
probs_positive = probs[:, 1]

print("\n" + "=" * 60)
print("CLASSIFICATION METRICS")
print("=" * 60)

# Accuracy
accuracy = accuracy_score(labels, preds)
print(f"\nAccuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")

# Precision, Recall, F1
precision = precision_score(labels, preds, average='binary')
recall = recall_score(labels, preds, average='binary')
f1 = f1_score(labels, preds, average='binary')

print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

# ROC-AUC
roc_auc = roc_auc_score(labels, probs_positive)
print(f"\nROC-AUC:   {roc_auc:.4f}")

# Confusion Matrix
print("\n" + "=" * 60)
print("CONFUSION MATRIX")
print("=" * 60)
cm = confusion_matrix(labels, preds)
print("\n              Predicted")
print("              0      1")
print(f"Actual 0   {cm[0,0]:5d}  {cm[0,1]:5d}")
print(f"       1   {cm[1,0]:5d}  {cm[1,1]:5d}")

# Calculate additional metrics from confusion matrix
tn, fp, fn, tp = cm.ravel()
specificity = tn / (tn + fp)
print(f"\nTrue Negatives:  {tn}")
print(f"False Positives: {fp}")
print(f"False Negatives: {fn}")
print(f"True Positives:  {tp}")
print(f"\nSpecificity: {specificity:.4f}")

# Detailed Classification Report
print("\n" + "=" * 60)
print("DETAILED CLASSIFICATION REPORT")
print("=" * 60)
print("\n" + classification_report(labels, preds, target_names=['Real', 'AI-Generated'], digits=4))

print("=" * 60)

COMPREHENSIVE MODEL EVALUATION

Computing predictions on validation set...


100%|██████████████████████████████████████████████████████████████████| 157/157 [01:22<00:00,  1.90it/s]


IndexError: index 1 is out of bounds for axis 1 with size 1

: 

In [14]:
# Visualize ROC Curve and Save Metrics

import matplotlib.pyplot as plt

# Plot ROC Curve
fpr, tpr, thresholds = roc_curve(labels, probs_positive)

plt.figure(figsize=(10, 8))
plt.plot(fpr, tpr, color='blue', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='red', lw=2, linestyle='--', label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('Receiver Operating Characteristic (ROC) Curve', fontsize=14, fontweight='bold')
plt.legend(loc="lower right", fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curve.png', dpi=300, bbox_inches='tight')
print("ROC curve saved to: roc_curve.png")
plt.show()

# Plot Confusion Matrix Heatmap
plt.figure(figsize=(8, 6))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.colorbar()

tick_marks = np.arange(2)
plt.xticks(tick_marks, ['Real', 'AI-Generated'], fontsize=11)
plt.yticks(tick_marks, ['Real', 'AI-Generated'], fontsize=11)

# Add text annotations
thresh = cm.max() / 2
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, format(cm[i, j], 'd'),
                ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black",
                fontsize=16, fontweight='bold')

plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=300, bbox_inches='tight')
print("Confusion matrix saved to: confusion_matrix.png")
plt.show()

# Save metrics to a file
metrics_dict = {
    'accuracy': accuracy,
    'precision': precision,
    'recall': recall,
    'f1_score': f1,
    'roc_auc': roc_auc,
    'specificity': specificity,
    'true_negatives': int(tn),
    'false_positives': int(fp),
    'false_negatives': int(fn),
    'true_positives': int(tp)
}

metrics_df = pd.DataFrame([metrics_dict])
metrics_df.to_csv('model_metrics.csv', index=False)
print("\nMetrics saved to: model_metrics.csv")
print("\nMetrics Summary:")
print(metrics_df.to_string(index=False))

NameError: name 'labels' is not defined

In [40]:
# Export Validation Predictions to CSV

print("\n" + "=" * 60)
print("EXPORTING VALIDATION PREDICTIONS")
print("=" * 60)

# Get image names from validation dataset indices
val_indices = val_idx  # These are the indices from train_test_split
image_names = []

for idx in val_indices:
    # Get the image name from the original dataset
    image_name = dataset.labels.iloc[idx]['image_name']
    image_names.append(image_name)

# Create DataFrame with image names and probabilities
predictions_df = pd.DataFrame({
    'image_name': image_names,
    'probability_ai_generated': probs_positive,
    'predicted_label': preds,
    'true_label': labels
})

# Sort by probability (highest first)
predictions_df = predictions_df.sort_values('probability_ai_generated', ascending=False)

# Save to CSV
output_path = 'validation_predictions.csv'
predictions_df.to_csv(output_path, index=False)

print(f"\n✓ Predictions saved to: {output_path}")
print(f"✓ Total predictions: {len(predictions_df)}")
print(f"\nFirst 10 predictions (sorted by probability):")
print(predictions_df.head(10).to_string(index=False))


EXPORTING VALIDATION PREDICTIONS

✓ Predictions saved to: validation_predictions.csv
✓ Total predictions: 5000

First 10 predictions (sorted by probability):
              image_name  probability_ai_generated  predicted_label  true_label
86299cf70eb249ed8583.jpg                       1.0                1           1
483f4e51e7711ac042dd.jpg                       1.0                1           1
ef3cdfb4f817a042a5ce.jpg                       1.0                1           1
46d91de2d31cc33f0996.jpg                       1.0                1           1
6ea3d4e65e434166c7bc.jpg                       1.0                1           1
cc8a39a7c31c0ed38398.jpg                       1.0                1           1
768de58b1706c47434ff.jpg                       1.0                1           1
27f58e23a7522d13d5c3.jpg                       1.0                1           1
1e264c2696ad85e4961c.jpg                       1.0                1           1
82ba2b3f378ac9feaf3c.jpg                 

In [1]:
import os

# Check current working directory
print(f"Current working directory: {os.getcwd()}")

Current working directory: /workspace


In [7]:
# Fine-tuning continuation setup: 1-logit EfficientNet-B4 + baseline checkpoint
import os
import torch
import torch.nn as nn
import timm
from torch.cuda.amp import autocast, GradScaler
from sklearn.metrics import roc_auc_score

torch.backends.cudnn.benchmark = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = timm.create_model(
    "efficientnet_b4",
    pretrained=False,
    num_classes=1
).to(device)

checkpoint_candidates = [
    "effiecientnet_b4_baseline.pth",  # requested filename in prompt
    "efficientnet_b4_baseline.pth",   # existing file in this workspace
    "/workspace/ai-gen-detection/training/baseline/effiecientnet_b4_baseline.pth",
    "/workspace/ai-gen-detection/training/baseline/efficientnet_b4_baseline.pth",
]

checkpoint_path = next((p for p in checkpoint_candidates if os.path.exists(p)), None)
if checkpoint_path is None:
    raise FileNotFoundError(f"Could not find baseline checkpoint. Tried: {checkpoint_candidates}")

state_dict = torch.load(checkpoint_path, map_location=device)

# Baseline checkpoint was trained with num_classes=2; skip classifier weights for num_classes=1 fine-tune.
state_dict = {
    k: v for k, v in state_dict.items()
    if not k.startswith("classifier.")
}

missing_keys, unexpected_keys = model.load_state_dict(state_dict, strict=False)

print(f"Loaded checkpoint: {checkpoint_path}")
print(f"Missing keys: {len(missing_keys)} | Unexpected keys: {len(unexpected_keys)}")

criterion = nn.BCEWithLogitsLoss()
scaler = GradScaler(enabled=torch.cuda.is_available())

Loaded checkpoint: efficientnet_b4_baseline.pth
Missing keys: 2 | Unexpected keys: 0


In [14]:
# DataLoader tuning + batch size optimization for AMP on RTX 4060-class GPUs
from torch.utils.data import DataLoader

def make_finetune_loaders(batch_size):
    train_loader_ft = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=8,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=2,
    )
    val_loader_ft = DataLoader(
        val_dataset,
        batch_size=batch_size * 2,
        shuffle=False,
        num_workers=8,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=2,
    )
    return train_loader_ft, val_loader_ft

def try_batch_size(bs):
    if not torch.cuda.is_available():
        return bs <= 24
    try:
        torch.cuda.empty_cache()
        probe_loader, _ = make_finetune_loaders(bs)
        images, labels = next(iter(probe_loader))
        images = images.to(device, non_blocking=True)
        labels = labels.float().unsqueeze(1).to(device, non_blocking=True)

        model.train()
        model.zero_grad(set_to_none=True)
        with autocast(enabled=True):
            logits = model(images)
            loss = criterion(logits, labels)
        scaler.scale(loss).backward()
        model.zero_grad(set_to_none=True)

        del images, labels, logits, loss, probe_loader
        torch.cuda.empty_cache()
        return True
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            torch.cuda.empty_cache()
            return False
        raise

BATCH_SIZE_FT = BATCH_SIZE if "BATCH_SIZE" in globals() else 16
BATCH_SIZE_FT = 24

# if BATCH_SIZE_FT == 16:
#     # This is often possible with AMP on RTX 4060 laptops, but depends on transforms/resolution/model state.
#     for candidate_bs in [24, 28]:
#         if try_batch_size(candidate_bs):
#             BATCH_SIZE_FT = candidate_bs

print(f"Fine-tuning batch size selected: {BATCH_SIZE_FT}")
train_loader_ft, val_loader_ft = make_finetune_loaders(BATCH_SIZE_FT)

Fine-tuning batch size selected: 24


In [8]:
# Fine-tuning helpers for staged unfreezing, AMP training, and ROC-AUC validation
import numpy as np
from tqdm import tqdm

def set_stage2_trainable_layers(model_ref):
    for p in model_ref.parameters():
        p.requires_grad = False

    for block_idx in [5, 6]:
        for p in model_ref.blocks[block_idx].parameters():
            p.requires_grad = True

    for p in model_ref.classifier.parameters():
        p.requires_grad = True

def unfreeze_all_layers(model_ref):
    for p in model_ref.parameters():
        p.requires_grad = True

def train_epoch_bce(loader, optimizer):
    model.train()
    running_loss = 0.0

    for images, labels in tqdm(loader):
        images = images.to(device, non_blocking=True)
        labels = labels.float().unsqueeze(1).to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast(enabled=torch.cuda.is_available()):
            logits = model(images)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()

    return running_loss / len(loader)

@torch.no_grad()
def validate_roc_auc(loader):
    model.eval()

    all_probs = []
    all_labels = []

    for images, labels in tqdm(loader):
        images = images.to(device, non_blocking=True)

        with autocast(enabled=torch.cuda.is_available()):
            logits = model(images)

        probs = torch.sigmoid(logits).squeeze(1)
        all_probs.extend(probs.detach().cpu().numpy())
        all_labels.extend(labels.numpy())

    return roc_auc_score(np.array(all_labels), np.array(all_probs))

In [16]:
# Stage 2 fine-tuning (epochs 6-10): unfreeze blocks[5], blocks[6], classifier
set_stage2_trainable_layers(model)

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4,
    weight_decay=1e-4
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=5,
    eta_min=1e-6
)

for epoch in range(6, 11):
    print(f"\nStage 2 | Epoch {epoch}/15")
    train_loss = train_epoch_bce(train_loader_ft, optimizer)
    val_roc_auc = validate_roc_auc(val_loader_ft)
    print(f"Train Loss: {train_loss:.6f}")
    print(f"Validation ROC-AUC: {val_roc_auc:.6f}")
    print(f"Learning Rate: {optimizer.param_groups[0]['lr']:.2e}")
    scheduler.step()


Stage 2 | Epoch 6/15


100%|██████████████████████████████████████████████████████████████████| 105/105 [00:56<00:00,  1.86it/s]


Train Loss: 0.073809
Validation ROC-AUC: 0.995800
Learning Rate: 1.00e-04

Stage 2 | Epoch 7/15


100%|██████████████████████████████████████████████████████████████████| 105/105 [00:50<00:00,  2.07it/s]


Train Loss: 0.051137
Validation ROC-AUC: 0.995377
Learning Rate: 9.05e-05

Stage 2 | Epoch 8/15


...

Train Loss: 0.043791
Validation ROC-AUC: 0.995242
Learning Rate: 6.58e-05

Stage 2 | Epoch 9/15


100%|██████████████████████████████████████████████████████████████████| 105/105 [00:48<00:00,  2.17it/s]


Train Loss: 0.041153
Validation ROC-AUC: 0.995826
Learning Rate: 3.52e-05

Stage 2 | Epoch 10/15


100%|██████████████████████████████████████████████████████████████████| 105/105 [00:32<00:00,  3.26it/s]

Train Loss: 0.035907
Validation ROC-AUC: 0.995434
Learning Rate: 1.05e-05


In [17]:
# Save model weights after Stage 2 (before full unfreeze Stage 3)
stage2_save_path = "effnetb4_stage2.pth"
torch.save(model.state_dict(), stage2_save_path)
print(f"Saved Stage 2 model weights to: {stage2_save_path}")

Saved Stage 2 model weights to: effnetb4_stage2.pth


In [ ]:
# Debug Cell 1 - GPU monitoring instructions
# Run this before training so you can watch utilization in real time.
print("Open a terminal and run: nvidia-smi -l 1")
print("Monitor GPU utilization (%), memory usage, and power draw during Stage 3 training.")

In [19]:
# Debug Cell 2 - Dataset size check
# Confirms the number of training samples.
print("Train dataset size:", len(train_dataset))

Train dataset size: 45000


In [20]:
# Debug Cell 3 - Batch size check
# Verifies DataLoader batch size used by training.
print("Batch size:", train_loader.batch_size)

Batch size: 16


In [21]:
# Debug Cell 4 - Input resolution check
# Prints current input tensor shape: [batch, channels, height, width].
images, labels = next(iter(train_loader))
print("Batch tensor shape:", images.shape)

Batch tensor shape: torch.Size([16, 3, 380, 380])


In [22]:
# Debug Cell 5 - DataLoader speed test
# Measures how long it takes to load 200 batches from train_loader.
import time
start = time.time()
for i, (x, y) in enumerate(train_loader):
    if i == 200:
        break
print("200 batch load time:", time.time() - start)

200 batch load time: 65.78439807891846


In [ ]:
# Debug Cell 6 - Model forward benchmark
# Benchmarks pure model compute (forward pass) using synthetic input.
import torch, time
x = torch.randn(train_loader.batch_size, 3, images.shape[2], images.shape[3]).cuda()
torch.cuda.synchronize()
start = time.time()
for _ in range(100):
    with torch.cuda.amp.autocast():
        y = model(x)
torch.cuda.synchronize()
print("100 forward passes time:", time.time() - start)

In [ ]:
# Debug Cell 7 - Batch timing instrumentation snippet
# Copy this snippet inside your training loop to print batch time every 50 batches.
timing_snippet = '''
import time

# Place before the batch loop:
# loop_start = time.time()

for batch_idx, (images, labels) in enumerate(train_loader_ft):
    batch_start = time.time()

    # ... existing training step logic ...

    batch_time = time.time() - batch_start
    if batch_idx % 50 == 0:
        print(f"Batch {batch_idx}: {batch_time:.4f}s")
'''

print(timing_snippet)

# Debug Run Order (Stage 3 Slowdown)

1. Run Cell 30 to Cell 32 to confirm dataset size, batch size, and input resolution.
2. Run Cell 33 to test DataLoader speed.
3. Run Cell 34 to benchmark raw GPU/model compute speed.
4. Start training and monitor GPU using `nvidia-smi -l 1` in a terminal.
5. Use Cell 35 timing snippet output (inside the training loop) to track per-batch processing time.

In [9]:
# Stage 3 Prep 1 - Load Stage 2 checkpoint weights
import os

stage2_ckpt_candidates = [
    "effnetb4_stage2.pth",
    "/workspace/ai-gen-detection/training/baseline/effnetb4_stage2.pth",
]
stage2_ckpt_path = next((p for p in stage2_ckpt_candidates if os.path.exists(p)), None)
if stage2_ckpt_path is None:
    raise FileNotFoundError(f"Stage 2 checkpoint not found. Tried: {stage2_ckpt_candidates}")

stage2_state = torch.load(stage2_ckpt_path, map_location=device)
model.load_state_dict(stage2_state, strict=True)
model = model.to(device)
print(f"Loaded Stage 2 weights from: {stage2_ckpt_path}")

Loaded Stage 2 weights from: effnetb4_stage2.pth


In [10]:
# Stage 3 Prep 2 - Set Stage 3 input resolution to 320x320 and batch size to 24
from torchvision import transforms
from torch.utils.data import DataLoader

train_transform_stage3 = transforms.Compose([
    transforms.RandomResizedCrop(320),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(5),
    transforms.ColorJitter(0.1, 0.1, 0.1, 0.05),
    transforms.ToTensor(),
])

val_transform_stage3 = transforms.Compose([
    transforms.Resize(352),
    transforms.CenterCrop(320),
    transforms.ToTensor(),
])

train_dataset_stage3 = TransformedDataset(train_subset, train_transform_stage3)
val_dataset_stage3 = TransformedDataset(val_subset, val_transform_stage3)

BATCH_SIZE_FT = 24

train_loader_ft = DataLoader(
    train_dataset_stage3,
    batch_size=BATCH_SIZE_FT,
    shuffle=True,
    num_workers=8,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2,
)

# Keep validation batch at 24 as well to reduce crash risk during Stage 3 validation.
val_loader_ft = DataLoader(
    val_dataset_stage3,
    batch_size=BATCH_SIZE_FT,
    shuffle=False,
    num_workers=8,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2,
)

images_stage3, _ = next(iter(train_loader_ft))
print("Stage 3 batch size:", train_loader_ft.batch_size)
print("Stage 3 input tensor shape:", tuple(images_stage3.shape))

Stage 3 batch size: 24
Stage 3 input tensor shape: (24, 3, 320, 320)


In [11]:
# Stage 3 Prep 3 - Verify AMP is active
print("CUDA available:", torch.cuda.is_available())
print("GradScaler enabled:", scaler.is_enabled())

model.eval()
with torch.no_grad():
    sample_images, _ = next(iter(train_loader_ft))
    sample_images = sample_images[:2].to(device, non_blocking=True)

    with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
        sample_outputs = model(sample_images)

print("Autocast output dtype:", sample_outputs.dtype)
print("AMP active (fp16/bf16):", sample_outputs.dtype in (torch.float16, torch.bfloat16))
model.train()

CUDA available: True
GradScaler enabled: True
Autocast output dtype: torch.float16
AMP active (fp16/bf16): True


EfficientNet(
  (conv_stem): Conv2d(3, 48, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (bn1): BatchNormAct2d(
    48, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
    (drop): Identity()
    (act): SiLU(inplace=True)
  )
  (blocks): Sequential(
    (0): Sequential(
      (0): DepthwiseSeparableConv(
        (conv_dw): Conv2d(48, 48, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=48, bias=False)
        (bn1): BatchNormAct2d(
          48, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
          (drop): Identity()
          (act): SiLU(inplace=True)
        )
        (aa): Identity()
        (se): SqueezeExcite(
          (conv_reduce): Conv2d(48, 12, kernel_size=(1, 1), stride=(1, 1))
          (act1): SiLU(inplace=True)
          (conv_expand): Conv2d(12, 48, kernel_size=(1, 1), stride=(1, 1))
          (gate): Sigmoid()
        )
        (conv_pw): Conv2d(48, 24, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (b

In [12]:
# Stage 3 fine-tuning (epochs 11-15): unfreeze all layers
unfreeze_all_layers(model)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-5,
    weight_decay=1e-4
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=5,
    eta_min=1e-7
)

for epoch in range(11, 16):
    print(f"\nStage 3 | Epoch {epoch}/15")
    train_loss = train_epoch_bce(train_loader_ft, optimizer)
    val_roc_auc = validate_roc_auc(val_loader_ft)
    print(f"Train Loss: {train_loss:.6f}")
    print(f"Validation ROC-AUC: {val_roc_auc:.6f}")
    print(f"Learning Rate: {optimizer.param_groups[0]['lr']:.2e}")
    scheduler.step()


Stage 3 | Epoch 11/15


100%|██████████████████████████████████████████████████████████████████| 209/209 [00:47<00:00,  4.39it/s]


Train Loss: 0.054127
Validation ROC-AUC: 0.993735
Learning Rate: 1.00e-05

Stage 3 | Epoch 12/15


100%|██████████████████████████████████████████████████████████████████| 209/209 [00:43<00:00,  4.83it/s]


Train Loss: 0.046828
Validation ROC-AUC: 0.993514
Learning Rate: 9.05e-06

Stage 3 | Epoch 13/15


...

Train Loss: 0.047154
Validation ROC-AUC: 0.993655
Learning Rate: 6.58e-06

Stage 3 | Epoch 14/15


100%|██████████████████████████████████████████████████████████████████| 209/209 [00:42<00:00,  4.88it/s]


Train Loss: 0.044948
Validation ROC-AUC: 0.993025
Learning Rate: 3.52e-06

Stage 3 | Epoch 15/15


100%|██████████████████████████████████████████████████████████████████| 209/209 [00:44<00:00,  4.68it/s]

Train Loss: 0.045681
Validation ROC-AUC: 0.993093
Learning Rate: 1.05e-06


In [13]:
# Save final fine-tuned weights and report final ROC-AUC
save_path = "effnetb4_finetuned.pth"
torch.save(model.state_dict(), save_path)
print(f"Saved fine-tuned model to: {save_path}")

final_roc_auc = validate_roc_auc(val_loader_ft)
print(f"Final validation ROC-AUC: {final_roc_auc:.6f}")

Saved fine-tuned model to: effnetb4_finetuned.pth


100%|██████████████████████████████████████████████████████████████████| 209/209 [00:44<00:00,  4.65it/s]

Final validation ROC-AUC: 0.993093
